In [1]:
import os

os.environ["PYSPARK_SUBMIT_ARGS"] = (
    '--conf "spark.driver.extraJavaOptions=-Djava.security.manager=allow" '
    '--conf "spark.executor.extraJavaOptions=-Djava.security.manager=allow" '
    'pyspark-shell'
)

**Create Spark Session**

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Spark Basics Assignment") \
    .getOrCreate()

print("Spark Started Successfully")

Spark Started Successfully


**Load Superstore Dataset**

In [3]:
import pandas as pd
pdf = pd.read_csv(
    "C:/Users/hp/OneDrive/Desktop/Assignment/Assignment 5/data5/Superstore.csv", encoding="latin1"
)
df = spark.createDataFrame(pdf)
df.show(5)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset 

**Explore Dataset**

In [4]:
print(df.columns)

['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']


**check for incorrect or inconsistent data**

Remove Duplicates

In [5]:
df = df.dropDuplicates()
print(df.count())

9994


Check Missing Values

In [6]:
from pyspark.sql.functions import col,isnan,when,count

df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|Row ID|Order ID|Order Date|Ship Date|Ship Mode|Customer ID|Customer Name|Segment|Country|City|State|Postal Code|Region|Product ID|Category|Sub-Category|Product Name|Sales|Quantity|Discount|Profit|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|     0|       0|         0|        0|        0|          0|            0|      0|      0|   0|    0|          0|     0|         0|       0|           0|           0|    0|       0|       0|     0|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+



Remove Null Values

In [11]:
df = df.dropna()

Check for Duplicate Rows

In [12]:
print("Total Rows:", df.count())
print("Distinct Rows:", df.distinct().count())

Total Rows: 9994
Distinct Rows: 9994


Check for Negative Sales

In [13]:
df.filter(df["Sales"] < 0).show()

+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|Row ID|Order ID|Order Date|Ship Date|Ship Mode|Customer ID|Customer Name|Segment|Country|City|State|Postal Code|Region|Product ID|Category|Sub-Category|Product Name|Sales|Quantity|Discount|Profit|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+



Check for Negative Profit

In [14]:
df.filter(df["Profit"] < 0).show()

+------+--------------+----------+----------+--------------+-----------+-------------------+-----------+-------------+------------+-------------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+---------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|      Customer Name|    Segment|      Country|        City|        State|Postal Code| Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|   Profit|
+------+--------------+----------+----------+--------------+-----------+-------------------+-----------+-------------+------------+-------------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+---------+
|   836|CA-2016-165316| 7/23/2016| 7/27/2016|Standard Class|   JB-15400|   Jennifer Braxton|  Corporate|United States|       Tampa|      Florida|      33614|  South|TEC-MA-10004002|     Technol

Remove the negative profit

In [15]:
df = df.filter(df["Profit"] >= 0)

Check for Invalid Quantity

In [17]:
df.filter(df["Quantity"] <= 0).show()

+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|Row ID|Order ID|Order Date|Ship Date|Ship Mode|Customer ID|Customer Name|Segment|Country|City|State|Postal Code|Region|Product ID|Category|Sub-Category|Product Name|Sales|Quantity|Discount|Profit|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+



Check Distinct Values in Region

In [18]:
df.select("Region").distinct().show()

+-------+
| Region|
+-------+
|  South|
|Central|
|   East|
|   West|
+-------+



Check Distinct Values in category

In [19]:
df.select("Category").distinct().show()

+---------------+
|       Category|
+---------------+
|Office Supplies|
|      Furniture|
|     Technology|
+---------------+



Check Distinct Values in segment

In [20]:
df.select("Segment").distinct().show()

+-----------+
|    Segment|
+-----------+
|   Consumer|
|Home Office|
|  Corporate|
+-----------+



Check for Leading/Trailing Spaces

In [21]:
from pyspark.sql.functions import trim

df.filter(col("Region") != trim(col("Region"))).show()

+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|Row ID|Order ID|Order Date|Ship Date|Ship Mode|Customer ID|Customer Name|Segment|Country|City|State|Postal Code|Region|Product ID|Category|Sub-Category|Product Name|Sales|Quantity|Discount|Profit|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+



Check Missing Customer Names

In [24]:
df.filter(col("Customer Name").isNull()).show()

+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|Row ID|Order ID|Order Date|Ship Date|Ship Mode|Customer ID|Customer Name|Segment|Country|City|State|Postal Code|Region|Product ID|Category|Sub-Category|Product Name|Sales|Quantity|Discount|Profit|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+



**Filtering the Data**

Sales Greater Than 500

In [25]:
df.filter(df["Sales"] > 500).show()

+------+--------------+----------+----------+--------------+-----------+-----------------+-----------+-------------+--------------+--------------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|    Customer Name|    Segment|      Country|          City|         State|Postal Code| Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+-----------------+-----------+-------------+--------------+--------------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|   407|CA-2017-117457| 12/8/2017|12/12/2017|Standard Class|   KH-16510|    Keith Herrera|   Consumer|United States| San Francisco|    California|      94110|   West|TEC-CO-10004115|     Techno

Region = West

In [26]:
df.filter(df["Region"] == "West").show()

+------+--------------+----------+----------+--------------+-----------+----------------+-----------+-------------+-------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|   Customer Name|    Segment|      Country|         City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+----------------+-----------+-------------+-------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|   407|CA-2017-117457| 12/8/2017|12/12/2017|Standard Class|   KH-16510|   Keith Herrera|   Consumer|United States|San Francisco|California|      94110|  West|TEC-CO-10004115|     Technology|     Copiers|Sharp AL-1

**Aggregation**

Total Rows

In [27]:
df.count()

8123

Average Sales

In [28]:
from pyspark.sql.functions import avg

df.select(avg("Sales")).show()

+------------------+
|        avg(Sales)|
+------------------+
|225.10078856333834|
+------------------+



Maximum Sales

In [29]:
from pyspark.sql.functions import max

df.select(max("Sales")).show()

+----------+
|max(Sales)|
+----------+
|  17499.95|
+----------+



In [ ]:
Minimum Sales

In [30]:
from pyspark.sql.functions import min

df.select(min("Sales")).show()

+----------+
|min(Sales)|
+----------+
|      0.99|
+----------+



**Group By Analysis**

Sales by Region

In [31]:
from pyspark.sql.functions import sum

df.groupBy("Region") \
  .agg(sum("Sales").alias("Total_Sales")) \
  .show()

+-------+-----------------+
| Region|      Total_Sales|
+-------+-----------------+
|  South|300086.7200000001|
|Central|359957.2320000001|
|   East|517917.2280000001|
|   West|650532.5255000002|
+-------+-----------------+



Orders by Category

In [32]:
from pyspark.sql.functions import count

df.groupBy("Category") \
  .agg(count("*").alias("Total_Orders")) \
  .show()

+---------------+------------+
|       Category|Total_Orders|
+---------------+------------+
|Office Supplies|        5140|
|      Furniture|        1407|
|     Technology|        1576|
+---------------+------------+



**Advanced Concept (Basic)**
Wide Transformation:
Operations like groupBy require moving data across partitions.

Shuffle:
Data movement between partitions.
Too much shuffle can slow Spark jobs.

**Save Output**

In [44]:
pandas_df = clean_df.toPandas()
pandas_df.to_csv(
    "C:/Users/hp/OneDrive/Desktop/Assignment/Assignment 5/output5/cleaned_superstore.csv",
    index=False,
)
print("File saved successfully!")

File saved successfully!
